In [1]:
import akshare as ak

In [2]:
from datetime import datetime, timedelta
# 定义时间范围
now_date = datetime.now().strftime("%Y%m%d")
year_ago_date = (datetime.now() - timedelta(days=365)).strftime("%Y%m%d")
month_ago_date = (datetime.now() - timedelta(days=30)).strftime("%Y%m%d")
half_year_ago_date = (datetime.now() - timedelta(days=180)).strftime("%Y%m%d")

# 个股

In [8]:
ak.stock_individual_info_em(symbol="000001")

,item,value
0,总市值,213271040996.019989
1,流通市值,213267551176.470001
2,行业,银行
3,上市时间,19910403
4,最新,10.99
5,股票代码,000001
6,股票简称,平安银行
7,总股本,19405918198.0
8,流通股,19405600653.0


In [8]:
ak.stock_zh_a_hist(symbol="688323", period="daily", start_date=month_ago_date, end_date=now_date)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [3]:
import pandas as pd

code = 309130
# 获取概念版块中的股票信息数据
df = pd.read_csv(f"ths_concept_stocks_{code}.csv", dtype={"代码": str})


In [4]:
code_list = df['代码'].tolist()
# 去掉 30 开头的股票代码
code_list = [code for code in code_list if not code.startswith("30")]
# 去掉 688 开头的股票代码
code_list = [code for code in code_list if not code.startswith("688")]  
# 去掉 920 开头的股票代码
code_list = [code for code in code_list if not code.startswith("920")]


In [23]:
tmp = ak.stock_individual_info_em(symbol=code_list[0])
tmp[tmp['item']=='流通市值']['value'].values[0]

9600356800.15

三、实战建议：如何用流通市值优化你的共振分析？
✅ 步骤：
在筛选“当日涨幅前100”时，优先保留流通市值在合理区间（如50～800亿）的股票
计算共振指标时，用流通市值加权（而非总市值）
例如：概念指数本身常按流通市值加权，个股也应匹配
警惕两类极端：
流通市值 < 30亿：小心流动性风险、闪崩
流通市值 > 总市值的90%：说明限售股已解禁，可能面临抛压
🔍 小技巧：
若某股总市值很大，但流通市值很小（如刚上市的次新股、大股东高比例质押），
则其价格极易被操控——这类股票在概念炒作中常成“先锋”，但退潮时跌得最狠。

In [5]:
# 获取市值和流通市值
for code in code_list:
    info = ak.stock_individual_info_em(symbol=code)
    # 提取市值和流通市值
    market_cap = info[info['item']=='总市值']['value'].values[0]
    circulating_cap = info[info['item']=='流通市值']['value'].values[0]
    print(f"股票代码: {code}, 总市值: {market_cap}, 流通市值: {circulating_cap}")
    # 添加到 df
    df.loc[df['代码'] == code, '总市值'] = market_cap
    df.loc[df['代码'] == code, '流通市值'] = circulating_cap

# 保存更新后的 df
df.to_csv(f"ths_concept_stocks_{code}.csv", index=False)

# 过滤出流通市值在合理区间的股票
df = df[(df['流通市值'] >= 50e8) & (df['流通市值'] <= 800e8)]

股票代码: 002721, 总市值: 9600356800.15, 流通市值: 9600356800.15
股票代码: 600986, 总市值: 17922857447.100002, 流通市值: 17922857447.100002
股票代码: 000607, 总市值: 5129199986.4, 流通市值: 4460568557.76
股票代码: 601615, 总市值: 48961403684.899994, 流通市值: 48961403684.899994
股票代码: 002405, 总市值: 26646092669.36, 流通市值: 26479767701.56
股票代码: 002115, 总市值: 12481156599.48, 流通市值: 11576289237.48
股票代码: 000156, 总市值: 16287276165.179998, 流通市值: 14941908958.829998
股票代码: 002195, 总市值: 59597529854.96, 流通市值: 58869260593.119995
股票代码: 002995, 总市值: 4049511233.6, 流通市值: 2565360686.56
股票代码: 002373, 总市值: 20969097613.05, 流通市值: 18280674052.02
股票代码: 600728, 总市值: 15397366664.42, 流通市值: 15397366664.42
股票代码: 600869, 总市值: 25189653667.1, 流通市值: 25189653667.1
股票代码: 600998, 总市值: 28136983905.72, 流通市值: 28136983905.72
股票代码: 002398, 总市值: 4474420298.08, 流通市值: 3649390101.89
股票代码: 600850, 总市值: 20705249676.04, 流通市值: 20705249676.04
股票代码: 600335, 总市值: 10545310278.6, 流通市值: 10545310278.6
股票代码: 002429, 总市值: 45903177754.98, 流通市值: 45876736802.520004
股票代码: 600522, 总市值: 67542273613

In [6]:
# 根据流通市值过滤股票
df = df[(df['流通市值'] >= 50e8) & (df['流通市值'] <= 800e8)]

In [7]:
df.to_csv('filter_stock_result.csv', index=False)

# 概念版块

In [2]:
concept_name = '商业航天'

In [3]:
tmp = ak.stock_board_concept_name_ths()
tmp[tmp['name']==concept_name]['code'].values[0]


  0%|          | 0/39 [00:00<?, ?it/s]

'309130'

In [7]:
ak.stock_board_concept_index_ths(symbol=concept_name, start_date=month_ago_date, end_date=now_date)


  0%|          | 0/2 [00:00<?, ?it/s]

,日期,开盘价,最高价,最低价,收盘价,成交量,成交额
0,2025-12-26,2220.822,2254.215,2213.413,2241.285,19824602000,4.412042e+11
1,2025-12-29,2237.283,2259.516,2229.294,2251.487,17401043000,4.045678e+11
2,2025-12-30,2235.576,2266.148,2229.192,2237.348,18803347000,4.395965e+11
3,2025-12-31,2239.949,2283.775,2228.247,2276.376,18104624000,4.245002e+11
4,2026-01-05,2300.417,2322.342,2273.995,2317.499,20045595000,5.014099e+11
5,2026-01-06,2310.781,2364.817,2307.754,2364.817,21195735000,5.153780e+11
6,2026-01-07,2357.075,2388.894,2339.992,2383.753,22313188000,5.430345e+11
7,2026-01-08,2375.655,2469.468,2374.706,2468.678,23987509000,5.766410e+11
8,2026-01-09,2492.799,2563.820,2488.956,2536.134,29207287000,7.291413e+11
9,2026-01-12,2581.768,2708.396,2581.465,2696.092,29241613000,7.541291e+11


In [7]:
ak.stock_board_concept_info_ths(symbol="阿里巴巴概念")

,项目,值
0,今开,2270.54
1,昨收,2263.61
2,最低,2263.20
3,最高,2302.05
4,成交量(万手),17233.70
5,板块涨幅,1.62%
6,涨幅排名,185/390
7,涨跌家数,276/62
8,资金净流入(亿),25.04
9,成交额(亿),3137.44
